# Tagged vs. Untagged Feature Comparison

This notebook compares the availability and distribution of machine-learning features between directly tagged buildings used for model training and untagged buildings that require prediction. The analysis is used to identify features whose availability may introduce sampling bias or reduce model generalizability.

**Input**
- Feature-engineered building data containing tagged and untagged buildings.

**Output**
- Feature-completeness and distribution comparisons between tagged and untagged buildings.
- Identification of potentially biased or poorly generalizable predictors for feature selection.

In [ ]:
import pandas as pd
import geopandas as gpd
import os
import warnings
warnings.filterwarnings('ignore')


ROOT_DIR  = '/fast/home/o-olajuyigbe/osm_project'
DATA_DIR    = os.path.join(ROOT_DIR, 'data')
PROC_DIR  = os.path.join(DATA_DIR, 'processed')

# building file needed to validate that all building IDs are present in the final output
PATH_FILE = os.path.join(PROC_DIR, 'germany_buildings_feature_engineered.parquet')

In [2]:
gdf = gpd.read_parquet(PATH_FILE)

In [3]:

# ---------------------------------------------------------------
# 2. LOAD DATA
# ---------------------------------------------------------------
# Using geopandas since the feature-engineered file carries a geometry column.
# We don't need geometry for this audit, but read_parquet via geopandas avoids
# breaking on the geometry dtype the way plain pandas.read_parquet sometimes does.
print(f"Loaded shape: {gdf.shape}")
print(gdf.columns.tolist())

# ---------------------------------------------------------------
# 3. DEFINE TAGGED vs UNTAGGED  <-- VERIFY THIS BLOCK AGAINST YOUR REAL SCHEMA
# ---------------------------------------------------------------
# From your week4 notes: building=yes means untagged/unclassified, anything
# else in the raw 'building' tag column means it carries a real type.
# If you already built a clean flag column (e.g. 'is_tagged') during feature
# engineering, just use that directly instead of recomputing it here.

LABEL_COL = 'stage1_l1'        # <-- change to whatever column holds the raw OSM building tag value

if 'is_tagged' in gdf.columns:
    tagged_mask = gdf['is_tagged'].astype(bool)
else:
    tagged_mask = gdf[LABEL_COL].notna()

n_tagged = tagged_mask.sum()
n_untagged = (~tagged_mask).sum()
print(f"Tagged:   {n_tagged:,} ({n_tagged/len(gdf)*100:.1f}%)")
print(f"Untagged: {n_untagged:,} ({n_untagged/len(gdf)*100:.1f}%)")



Loaded shape: (38802372, 159)
['id', 'geometry', 'building', 'tag_l1', 'tag_l2', 'is_abandoned', 'tag_is_mixed', 'tag_source', 'tag_used', 'all_candidates', 'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street', 'name', 'amenity', 'building:use', 'craft', 'office', 'shop', 'tags', 'address_tier', 'compactness', 'building_l1', 'building_l2', 'building_is_mixed', 'buse_l1', 'buse_l2', 'buse_is_mixed', 'amenity_l1', 'amenity_l2', 'shop_l1', 'shop_l2', 'stage1_l1', 'stage1_l2', 'stage1_is_mixed', 'stage1_source', 'raw_label', 'stage2_l1', 'stage2_l2', 'stage2_source', 'landuse_l1', 'landuse_l2', 'landuse_used', 'landuse_osm_id', 'has_building_levels', 'building_levels', 'has_name', 'area', 'perimeter', 'convexity', 'num_vertices', 'mrr_long', 'mrr_short', 'elongation', 'rectangularity', 'diameter', 'dist_nearest_building', 'is_touching_building', 'touching_building_count', 'dist_nearest_real_building', 'is_touching_real_building', 'real_touching_count', 'dist_to_residential', 'di

In [4]:
DROP_IDENTIFIERS = ['geometry',
    'addr:city', 'addr:housenumber', 'addr:postcode', 'addr:street',
    'name', 'tags', 'id'
]

# Raw OSM tag columns — already encoded into building_l1/l2, buse_l1/l2 etc.
DROP_RAW_TAGS = [
    'building', 'amenity', 'building:use', 'craft', 'office', 'shop',
]

# Pipeline provenance — tells you HOW a label was assigned, not what the building is
# Useful for debugging but leaks information into ML if used as features
DROP_PIPELINE = [
    'stage1_source', 'stage1_is_mixed',
    'stage2_source', 'is_abandoned',
    'landuse_used', 'landuse_osm_id',
    'buse_is_mixed', 'building_is_mixed',
    'tag_source', 'tag_used', 'tag_is_mixed',
    'all_candidates', 'raw_label',
]

# Intermediate label columns — these are partial/noisy versions of the target
DROP_LABEL_LEAKAGE = [  
    'building_l1', 'building_l2',
    'buse_l1', 'buse_l2',
    'amenity_l1', 'amenity_l2',
    'shop_l1', 'shop_l2',
    'landuse_l1', 'landuse_l2',   # direct leakage — landuse IS what you're predicting
    'tag_l1', 'tag_l2',
]

# Targets — remove from features
TARGETS = ['stage1_l1', 'stage1_l2', 'stage2_l1', 'stage2_l2',] # targets not dropped here

ALL_DROP = (
    DROP_IDENTIFIERS + DROP_RAW_TAGS +
    DROP_PIPELINE + DROP_LABEL_LEAKAGE + TARGETS
)

# Only drop columns that actually exist
exclude_cols = [c for c in ALL_DROP if c in gdf.columns]

# Feature columns = everything except targets
feature_cols = [c for c in gdf.columns if c not in exclude_cols]

In [5]:
tagged = gdf[tagged_mask]
untagged = gdf[~tagged_mask]

print(tagged.shape)
print(untagged.shape)

(13299497, 159)
(25502875, 159)


#### Completeness Check 
Check if the features are equally available in tagged and untagged buildings?

In [13]:
results = []

for col in feature_cols:

    tagged_pct = tagged[col].notna().mean() * 100
    untagged_pct = untagged[col].notna().mean() * 100

    diff = tagged_pct - untagged_pct

    results.append({
        'feature': col,
        'tagged_%': tagged_pct,
        'untagged_%': untagged_pct,
        'difference_%': diff
    })

completeness_df = pd.DataFrame(results)

completeness_df = (
    completeness_df
    .sort_values('difference_%', ascending=False)
    .round(2)
)

completeness_df.head(5)

,feature,tagged_%,untagged_%,difference_%
3,building_levels,24.87,1.37,23.51
0,address_tier,100.00,100.00,0.00
1,compactness,100.00,100.00,0.00
2,has_building_levels,100.00,100.00,0.00
4,has_name,100.00,100.00,0.00


#### Binary Check
Check mean

In [15]:
binary_cols = [
    c for c in feature_cols
    if str(gdf[c].dtype) in ['int8', 'bool']
]
binary_results = []

for col in binary_cols:
    tagged_rate = tagged[col].mean() * 100
    untagged_rate = untagged[col].mean() * 100
    abs_diff = abs(tagged_rate - untagged_rate)
    rel_diff = (
        abs_diff /
        max(tagged_rate, 0.0001)
    ) * 100
    ratio = (
        tagged_rate / max(untagged_rate, 0.001)
    )

    binary_results.append({
        'feature': col,
        'tagged_%': tagged_rate,
        'untagged_%': untagged_rate,
        'abs_diff_pp': abs_diff,      # percentage points
        'rel_diff_%': rel_diff,        # percent decrease
        'ratio': ratio
    })

binary_df = pd.DataFrame(binary_results)

binary_df.sort_values(
    'rel_diff_%',
    ascending=False
).head(30)

,feature,tagged_%,untagged_%,abs_diff_pp,rel_diff_%,ratio
0,has_building_levels,24.874317,1.366289,23.508028,94.507230,18.205749
1,has_name,3.500584,0.519314,2.981270,85.164930,6.740784
14,near_light_rail_200m,5.763143,1.745388,4.017755,69.714657,3.301927
11,near_pedestrian_zone_25m,0.205496,0.100706,0.104790,50.993668,2.040553
5,real_touching_count,49.937340,66.037217,16.099878,32.240159,0.756200
3,touching_building_count,85.072481,68.925068,16.147412,18.980771,1.234275
9,near_service_road_35m,41.032988,34.311006,6.721982,16.381897,1.195913
15,near_heavy_rail_500m,28.761659,24.258810,4.502849,15.655735,1.185617
6,near_major_road_700m,39.507863,33.629828,5.878035,14.878139,1.174786
8,near_residential_road_25m,56.494881,48.504273,7.990608,14.143951,1.164740


#### Numeric Features

In [17]:
numeric_cols = gdf[feature_cols].select_dtypes(
    include=['int16','int32','int64',
             'float32','float64']
).columns

numeric_results = []

for col in numeric_cols:

    tagged_mean = tagged[col].mean()
    untagged_mean = untagged[col].mean()

    tagged_std = tagged[col].std()

    shift = abs(
        tagged_mean - untagged_mean
    ) / (tagged_std + 1e-9)

    numeric_results.append({
        'feature': col,
        'shift_score': shift
    })

shift_df = pd.DataFrame(numeric_results)

shift_df.sort_values(
    'shift_score',
    ascending=False
).head(50)

,feature,shift_score
53,dist_nearest_education,0.463471
79,dist_nearest_pedestrian_zone,0.412601
15,dist_to_commercial,0.409306
58,dist_nearest_transport,0.372529
96,dist_nearest_light_rail,0.371147
86,road_count_total_200m,0.327791
0,address_tier,0.326885
98,dist_nearest_heavy_rail,0.287953
84,road_count_service_road_200m,0.276382
2,building_levels,0.274481


In [22]:
add_tagged_rate = tagged['address_tier'].mean() * 100
add_untagged_rate = untagged['address_tier'].mean() * 100
add_abs_diff = abs(tagged_rate - untagged_rate)
add_rel_diff = (
        abs_diff /
        max(tagged_rate, 0.0001)
    ) * 100
add_ratio = (
        tagged_rate / max(untagged_rate, 0.001)
    )

print(f'tagged_rate: {add_tagged_rate}, \nuntagged_rate: {add_untagged_rate}, \nabs_diff: {add_abs_diff}, \nrel_diff: {add_rel_diff}, \nratio: {add_ratio}')

tagged_rate: 101.0962595051527, 
untagged_rate: 69.22684207172722, 
abs_diff: 31.869417433425482, 
rel_diff: 31.52383440240057, 
ratio: 1.4603621439268455
